# Semaine 3 - Jours 1 & 2 : Architectures Multi-Agents & Model Context Protocol (MCP)

## Objectifs pédagogiques
- Comprendre les limites d'un agent unique face à des processus métiers complexes.
- Maîtriser les principaux design patterns d'orchestration multi-agents.
- Assimiler l'architecture et l'intérêt stratégique du protocole MCP (Model Context Protocol).
- Implémenter des interactions agents-outils standardisées et découplées.

---

## Table des Matières
1. **Jour 1 : L'Émergence du Multi-Agent**
   - 1.1 Limites des agents uniques
   - 1.2 Catalogue des Patterns d'Architecture
   - 1.3 Exercice 1 : Modélisation d'une architecture multi-agent
2. **Jour 2 : Model Context Protocol (MCP)**
   - 2.1 L'écosystème MCP (Client, Serveur, Resources, Tools, Prompts)
   - 2.2 Cas d'usage entreprise & alignement technologique
   - 2.3 Exercice 2 : Schéma d'intégration MCP & Snippet de code

# Jour 1 : Pourquoi les architectures multi-agents émergent

### 1.1 Limites des agents uniques
Lorsqu'un agent unique (Single Agent System) est confronté à des tâches complexes, on observe rapidement plusieurs points de rupture familiers des architectes logiciels :
- **Explosion du contexte (Prompt Bloat)** : Accumuler toutes les instructions, règles métiers et descriptions d'outils dans un seul système engendre une perte d'attention de l'IHM sous-jacente (*Lost in the Middle*).
- **Manque de robustesse (Single Point of Failure)** : Si l'agent échoue sur une sous-tâche ou hallucine sur l'appel d'un outil, l'ensemble de la transaction s'effondre.
- **Difficulté de maintenance** : Un agent généraliste est l'équivalent d'une classe « Dieu » (God Class) en programmation orientée objet. Il est impossible à tester unitairement et difficile à faire évoluer.

**La solution ?** Appliquer les principes SOLID et le découplage en micro-services au monde des LLM en distribuant les responsabilités à des agents spécialisés.

### 1.2 Catalogue des Patterns d'Architecture

#### A. Planner / Executor
Un premier agent (le Planificateur) reçoit la requête utilisateur et décompose la stratégie en une suite d'étapes ordonnées. Un ou plusieurs agents (les Exécuteurs) réalisent les tâches de manière séquentielle.

#### B. Supervisor
Un agent central (le Superviseur) agit comme un orchestrateur de workflow (façon BPMN). Il analyse l'état courant et route dynamiquement la parole ou la tâche à l'agent spécialisé le plus pertinent, puis récupère le résultat pour décider de la suite.

#### C. Swarm (Essaim)
Pattern popularisé par OpenAI. Il n'y a pas de superviseur central permanent. Les agents s'échangent la main de manière fluide via un mécanisme de transfert (*handoff*). Un agent décide lui-même, en fonction du contexte, de passer le contrôle à un autre agent.

#### D. Hierarchical Agents (Structure Hiérarchique)
Adapté aux grandes entreprises. Un super-superviseur orchestre des sous-équipes d'agents. Chaque sous-équipe possède son propre superviseur local caché pour le niveau supérieur. Idéal pour encapsuler des domaines fonctionnels (Domain-Driven Design).

### Schémas d'Architecture des Patterns

```
1. PLANNER / EXECUTOR
[User] -> [ Planner ]
               | (Génère la roadmap)
               v
         [ Executor 1 ] -> [ Executor 2 ] -> [ Retour User ]

2. SUPERVISOR
         +---> [ Agent Codeur ] ---+
         |                         |
[User] <-> [ Agent Superviseur ] <--+ (Centralise et valide)
         |                         |
         +---> [ Agent Testeur ] --+

3. SWARM (Handoff)
[User] -> [ Agent Tri ] --(handoff)--> [ Agent Facturation ] --(handoff)--> [ Agent Support ]
```

### Notes Pédagogiques & Correspondances Frameworks
> - **OpenAI Agents SDK** : Conçu nativement autour du pattern **Swarm** avec des concepts de `Agent` et de `handoff` (fonctions qui retournent une autre instance d'agent).
> - **LangGraph (LangChain)** : Idéal pour les patterns **Supervisor** et **Hierarchical**. Il permet de modéliser les interactions sous forme de graphes d'états cycliques ou acycliques avec un contrôle fin de la State Transition.
> - **AutoGen (Microsoft)** : Historiquement fort sur les patterns de conversations multi-agents complexes et les simulations de groupes (ConversableAgent).

## 🛠️ Exercice 1 : Conception d'Architecture Multi-Agent

**Scénario de l'exercice :**
Une direction financière souhaite automatiser l'analyse des rapports de performance trimestriels. Le processus demande :
1. L'extraction des données brutes depuis une base de données.
2. La vérification de la cohérence comptable (calculs des ratios).
3. La rédaction d'une note de synthèse macro-économique.
4. La validation finale par rapport aux directives d'un guide de style d'entreprise.

**Votre tâche :**
En tant qu'architecte, décrivez le pattern le plus adapté pour répondre à ce besoin. Justifiez votre choix, listez les agents requis ainsi que leur rôle respectif.

### 📝 [Espace pour votre réponse - Exercice 1]
*Double-cliquez ici pour saisir votre analyse d'architecture (Patterns, Agents, Flux de données).* 

- **Pattern choisi :** ...
- **Justification :** ...
- **Définition des Agents & Responsabilités :** ...

# Jour 2 : Model Context Protocol (MCP)

### 2.1 L'écosystème MCP
Le **Model Context Protocol (MCP)** est un protocole ouvert (initié par Anthropic) qui standardise la manière dont les applications d'IA se connectent aux données et aux outils. 

Avant MCP, chaque framework (LangChain, AutoGen, LlamaIndex) avait sa propre abstraction pour définir un outil ou une source de données. Avec MCP, on applique le même paradigme que le *Language Server Protocol (LSP)* de VS Code : **écrire le connecteur une fois, l'utiliser partout**.

#### Composants Clés :
- **MCP Client** : L'application hôte (ex: Claude Desktop, votre application backend Python/Java, ou un orchestrateur multi-agent) qui initie la session.
- **MCP Server** : Un microservice léger qui expose des capacités spécifiques via le protocole MCP.
- **Resources** : Données textuelles ou binaires en lecture seule que le serveur met à disposition de l'IA (ex: fichiers, logs, schémas de BDD).
- **Tools** : Fonctions exécutables (fonctions avec des effets de bord) que l'IA peut appeler (ex: écrire un fichier, exécuter une requête SQL, envoyer un e-mail).
- **Prompts** : Modèles de prompts préconfigurés et paramétrables fournis directement par le serveur.

### Architecture d'intégration MCP en Entreprise

```
+-------------------------------------------------------------------------+
|                           APPLICATION COEUR                             |
|  [ Orchestrateur Multi-Agent (LangGraph / OpenAI SDK / Custom) ]       |
|                           ^               ^                             |
|                           | (Session MCP) |                             |
|                           v               v                             |
|                    [ MCP Client 1 ]   [ MCP Client 2 ]                  |
+---------------------------|---------------|-----------------------------+
                            |               | (Transport : Stdio / SSE)
                            v               v
               +------------------+   +------------------+
               |    MCP SERVER    |   |    MCP SERVER    |
               |  (PostgreSQL)    |   |   (Enterprise)   |
               |  - Tool: query   |   |  - Resource: CRM |
               +------------------+   +------------------+
```

### 2.2 Cas d'usage entreprise & Alignement technologique
Pour une architecture d'entreprise, MCP offre des avantages majeurs :
1. **Sécurité et Isolation** : Le serveur MCP s'exécute dans son propre processus (ou conteneur). L'accès aux bases de données de production ou aux API internes est encapsulé et auditable.
2. **Gouvernance des outils** : Les équipes Data ou SecOps peuvent mettre à disposition un catalogue de serveurs MCP certifiés sans se soucier du framework de build de l'équipe IA.
3. **Interopérabilité Polyglotte** : Un serveur MCP peut être développé en TypeScript ou Python, tandis que votre application principale s'exécute sur une JVM (via l'implémentation MCP Java naissante).

In [ ]:
# Code d'illustration : Exemple théorique d'initialisation d'un client MCP en Python
# Ce bloc montre la standardisation de l'initialisation d'un serveur tiers.

import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def run_mcp_client():
    # Configuration des paramètres pour lancer un serveur MCP PostgreSQL (via Node/npx par exemple)
    server_params = StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-postgres", "postgresql://localhost:5432/mydb"]
    )
    
    async with stdio_client(server_params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            # Initialisation de la connexion
            await session.initialize()
            
            # Liste des outils découverts automatiquement
            tools = await session.list_tools()
            print("--- Outils MCP découverts ---")
            for tool in tools.tools:
                print(f"Outil: {tool.name} - Description: {tool.description}")

# Note : Pour l'exécuter dans un notebook, décommentez la ligne ci-dessous
# asyncio.run(run_mcp_client())

## 🛠️ Exercice 2 : Conception de Solution MCP

**Scénario de l'exercice :**
Vous devez concevoir un système où un agent IA doit interagir avec le système de ticketing d'un support client (ex: Jira ou ServiceNow) pour récupérer les incidents critiques et les résumer.

**Votre tâche :**
1. Déterminez ce qui relève d'une **Resource** et ce qui relève d'un **Tool** dans ce contexte.
2. Proposez la structure JSON du schéma de définition d'un outil nommé `create_incident` conforme aux standards de déclaration de fonctions.

### 📝 [Espace pour votre réponse - Exercice 2]
*Double-cliquez ici pour saisir votre découpage technique MCP.*

#### 1. Identification MCP
- **Resources proposées :** ...
- **Tools proposés :** ...

#### 2. Structure JSON (Schéma de l'outil `create_incident`)

In [ ]:
# Remplissez ici le dictionnaire Python simulant le schéma JSON du Tool
tool_definition = {
    "name": "create_incident",
    "description": "Crée un nouvel incident dans le système de support technique",
    "inputSchema": {
        "type": "object",
        "properties": {
            # À vous de compléter les propriétés attendues pour un incident (ex: title, priority)
        },
        "required": []
    }
}

### Références & Lectures recommandées
- [Specification Model Context Protocol (Anthropic)](https://modelcontextprotocol.io)
- [OpenAI Agents SDK - Official Repository](https://github.com/openai/openai-agents)
- [LangGraph : Concepts d'orchestration multi-agents](https://langchainai.github.io/langgraph/)
- [Microsoft AutoGen : Multi-Agent Conversation Framework](https://microsoft.github.io/autogen/)

C'est une excellente démarche. Pour un profil d'architecte et de développeur backend senior, la validation des modèles d'architecture et des schémas de données permet de s'assurer que les concepts de découplage et de robustesse sont bien appliqués.

Voici les solutions détaillées et expliquées pour les deux exercices intégrés au premier notebook (**`S3_J1_J2_MultiAgent_MCP.ipynb`**).

---

## 🛠️ Solution de l'Exercice 1 : Conception d'Architecture Multi-Agent

### 📝 Rappel du scénario

Automatiser l'analyse des rapports financiers trimestriels :

1. Extraction des données brutes (Base de données).
2. Vérification de la cohérence comptable (Calculs de ratios).
3. Rédaction d'une note de synthèse macro-économique (Analyse textuelle/contexte).
4. Validation finale (Respect du guide de style de l'entreprise).

---

### 1. Pattern d'architecture recommandé : **Supervisor (Superviseur) ou Hiérarchique**

#### **Justification du choix (Vision Architecte) :**

Le pattern **Planner/Executor** pourrait fonctionner, mais il est trop rigide : si l'étape de vérification comptable (étape 2) échoue ou détecte une anomalie, un planificateur linéaire ne saura pas facilement "recommencer" ou demander une correction.
Le pattern **Swarm** (Essaim) laisse trop d'autonomie aux agents. Dans un contexte financier et de conformité d'entreprise, nous avons besoin d'un **déterminisme de workflow** et d'une gouvernance stricte.

Le pattern **Supervisor** est donc le plus adapté. L'agent Superviseur agit comme un orchestrateur de processus (similaire à un moteur de workflow BPMN ou un orchestrateur de microservices). Il gère l'état global (*State*), distribue les tâches de manière séquentielle ou parallèle, et valide les critères de passage à l'étape suivante (*Quality Gates*).

---

### 2. Définition des Agents & Responsabilités

* **Agent 1 : L'Orchestrateur / Superviseur (Supervisor Agent)**
* **Rôle :** Reçoit la requête, maintient le statut du rapport, distribue le travail aux agents spécialisés et valide leur feedback.
* **Prompt/Instruction clé :** *"Tu es le garant du workflow. Tu passes la main à l'Agent Data, puis tu envoies le résultat à l'Agent Analyste. Si l'Agent Validateur rejette le rapport, tu demandes à l'Agent Rédacteur de corriger."*


* **Agent 2 : L'Extracteur de Données (Data Extraction Agent)**
* **Rôle :** Traduit le besoin en requêtes techniques (via des outils/MCP) pour extraire les données financières brutes.
* **Outils associés :** Outil d'accès à la base de données SQL ou aux API financières.


* **Agent 3 : L'Analyste Comptable (Comptability & Ratios Agent)**
* **Rôle :** Reçoit les données brutes, applique les formules mathématiques rigoureuses, calcule les ratios (EBITDA, ROI, etc.) et détecte les incohérences (ex: bilan non équilibré). Il ne rédige pas de texte, il valide des chiffres.


* **Agent 4 : Le Rédacteur Financier (Writer/Synthesizer Agent)**
* **Rôle :** Prend les ratios validés et le contexte macro-économique pour rédiger une note de synthèse claire, structurée et professionnelle.


* **Agent 5 : Le Contrôleur de Conformité / Validateur (Compliance & Style Agent)**
* **Rôle :** Reçoit la note rédigée. Il compare le document avec le guide de style et les contraintes légales de l'entreprise. Il émet un avis : `APPROVE` ou `REJECT` avec feedback.



---

### Flux de données et d'exécution (Diagramme séquentiel)

1. `User` ➔ Requête de génération ➔ `Superviseur`
2. `Superviseur` ➔ Demande d'extraction ➔ `Agent Data` ➔ (Retourne le JSON brut)
3. `Superviseur` ➔ Demande de calcul ➔ `Agent Analyste` ➔ (Retourne les ratios calculés)
4. `Superviseur` ➔ Demande de rédaction ➔ `Agent Rédacteur` ➔ (Retourne le draft de la note)
5. `Superviseur` ➔ Demande d'audit ➔ `Agent Validateur`
* *Si KO :* `Agent Validateur` retourne les corrections ➔ `Superviseur` renvoie à l' `Agent Rédacteur`.
* *Si OK :* Le `Superviseur` valide l'état final et répond à l'utilisateur.



---

## 🛠️ Solution de l'Exercice 2 : Conception de Solution MCP

### 📝 Rappel du scénario

Interaction avec un système de ticketing (Jira/ServiceNow) pour récupérer les incidents critiques, les résumer et en créer de nouveaux.

---

### 1. Identification MCP : Différence entre Resource et Tool

En architecture MCP, la frontière repose sur la notion **d'effet de bord** et de **dynamisme** :

* **Resources (Lecture seule, Contexte statique ou semi-dynamique) :**
* *La liste des utilisateurs de l'équipe support* (URI: `jira://teams/support/members`).
* *Le guide d'escalade des incidents* ou la documentation d'architecture de l'entreprise (URI: `enterprise://docs/incident-playbook`).
* *Les logs bruts d'un incident spécifique* (URI: `jira://incidents/INC-001/logs`).
* *Pourquoi ?* Ce sont des sources de données textuelles que le LLM va simplement lire pour comprendre la situation ou enrichir son prompt (approche RAG).


* **Tools (Actions, Effets de bord, Mutations) :**
* `create_incident` : Modifie l'état du système en insérant une nouvelle ligne en base.
* `assign_incident` : Change le propriétaire d'un ticket.
* `add_comment` : Ajoute une note dans le fil d'actualité du ticket.
* *Pourquoi ?* Ce sont des fonctions exécutables qui modifient l'état du SI (opérations de type écriture/mise à jour) et qui nécessitent des paramètres stricts.



---

### 2. Structure JSON (Schéma de l'outil `create_incident`)

Voici la structure JSON standardisée (conforme à la spécification MCP et JSON Schema) que le serveur MCP doit renvoyer lors de l'appel à `list_tools`.

```json
{
  "name": "create_incident",
  "description": "Crée un nouvel incident de support dans le système de ticketing de l'entreprise (Jira/ServiceNow).",
  "inputSchema": {
    "type": "object",
    "properties": {
      "title": {
        "type": "string",
        "description": "Résumé succinct et explicite du problème (ex: 'Crash de la JVM sur la passerelle de paiement')."
      },
      "project_key": {
        "type": "string",
        "description": "Le code du projet ou de l'équipe concernée (ex: 'JAVA', 'OPS', 'DATA')."
      },
      "severity": {
        "type": "string",
        "enum": ["LOW", "MEDIUM", "HIGH", "CRITICAL"],
        "description": "Le niveau d'urgence de l'anomalie défini par les règles de l'entreprise."
      },
      "description": {
        "type": "string",
        "description": "Détails techniques complets incluant les logs d'erreur, les traces de pile (stacktrace) ou les étapes de reproduction."
      },
      "reporter_email": {
        "type": "string",
        "format": "email",
        "description": "Adresse email de l'ingénieur ou du système automatisé ayant détecté l'erreur."
      }
    },
    "required": ["title", "project_key", "severity", "description"]
  }
}

```

### 💡 Note d'explication technique pour le backend :

* **L'usage de l'`enum**` sur le champ `severity` est crucial. Cela évite que le LLM invente des sévérités exotiques (comme `"URGENT"`, `"P1"`, `"BLOQUANT"`) qui feraient planter votre API de destination.
* **Le champ `required**` force l'orchestrateur ou le modèle à aller chercher ou à demander l'information manquante à l'utilisateur *avant* de tenter de déclencher l'outil, garantissant la cohérence des données à la frontière de votre système de ticketing.